# PneumoAI - Binary Pneumonia / Normal

Notebook Colab autonome pour revenir au modele binaire fiable base sur le dataset Kaggle Chest X-Ray Pneumonia.

Runtime recommande: GPU T4.

In [ ]:
!nvidia-smi
!pip -q install kaggle scikit-learn pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Option A: Kaggle username + key classique
KAGGLE_USERNAME = ""
KAGGLE_KEY = ""

# Option B: nouveau token Kaggle KGAT_xxx si disponible
KAGGLE_API_TOKEN = ""

os.makedirs('/root/.kaggle', exist_ok=True)
if KAGGLE_API_TOKEN:
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(KAGGLE_API_TOKEN)
else:
    import json
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle', 0o700)
!kaggle datasets list -s chest-xray-pneumonia | head

In [ ]:
import os, zipfile
from pathlib import Path

DATA_ROOT = Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --force

zip_path = next(DATA_ROOT.glob('*.zip'))
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(DATA_ROOT)

for root, dirs, files in os.walk(DATA_ROOT):
    if Path(root).name == 'chest_xray':
        print('Found:', root)
        break

In [ ]:
from pathlib import Path

candidates = list(DATA_ROOT.rglob('chest_xray'))
DATASET_DIR = None
for candidate in candidates:
    if (candidate / 'train' / 'NORMAL').exists() and (candidate / 'train' / 'PNEUMONIA').exists():
        DATASET_DIR = candidate
        break
if DATASET_DIR is None:
    raise FileNotFoundError('Dataset chest_xray introuvable apres extraction')

print('DATASET_DIR =', DATASET_DIR)
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        print(split, cls, len(list((DATASET_DIR / split / cls).glob('*'))))

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
FINE_TUNE_EPOCHS = 8
MODEL_PATH = '/content/multirun_pneumo_binary/pneumonia_binary.keras'
METRICS_PATH = '/content/multirun_pneumo_binary/binary_metrics.csv'
Path('/content/multirun_pneumo_binary').mkdir(parents=True, exist_ok=True)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=12,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.12,
    horizontal_flip=True,
    fill_mode='nearest'
)
eval_datagen = ImageDataGenerator(rescale=1./255)

common = dict(target_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, class_mode='binary')
train_gen = train_datagen.flow_from_directory(DATASET_DIR / 'train', shuffle=True, **common)
val_gen = eval_datagen.flow_from_directory(DATASET_DIR / 'val', shuffle=False, **common)
test_gen = eval_datagen.flow_from_directory(DATASET_DIR / 'test', shuffle=False, **common)
print(train_gen.class_indices)

In [ ]:
base_model = MobileNetV2(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.45)(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(base_model.input, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)
model.summary()

In [ ]:
labels = train_gen.classes
counts = np.bincount(labels)
class_weight = {i: len(labels) / (len(counts) * count) for i, count in enumerate(counts)}
print('class_weight =', class_weight)

callbacks = [
    EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
    ModelCheckpoint(MODEL_PATH, monitor='val_auc', mode='max', save_best_only=True)
]

history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, class_weight=class_weight, callbacks=callbacks)

In [ ]:
for layer in base_model.layers[:100]:
    layer.trainable = False
for layer in base_model.layers[100:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)
history_ft = model.fit(train_gen, validation_data=val_gen, epochs=FINE_TUNE_EPOCHS, class_weight=class_weight, callbacks=callbacks)

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)
test_gen.reset()
y_true = test_gen.classes
y_prob = best_model.predict(test_gen).ravel()
y_pred = (y_prob >= 0.5).astype('int32')

metrics = pd.DataFrame([{
    'accuracy': accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred, zero_division=0),
    'recall': recall_score(y_true, y_pred, zero_division=0),
    'f1': f1_score(y_true, y_pred, zero_division=0),
    'auc': roc_auc_score(y_true, y_prob),
    'threshold': 0.5
}])
metrics.to_csv(METRICS_PATH, index=False)
print(metrics)
print(confusion_matrix(y_true, y_pred))

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/PneumoAI_binary'
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
!cp /content/multirun_pneumo_binary/pneumonia_binary.keras "$DRIVE_DIR/pneumonia_binary.keras"
!cp /content/multirun_pneumo_binary/binary_metrics.csv "$DRIVE_DIR/binary_metrics.csv"
print('Saved to', DRIVE_DIR)